# 01. ETL Pipeline: PerfectPetClub

**Module:** 7COM2005 - Data Engineering  
**Institution:** University of Hertfordshire

### Objective
Extract, transform, and load data from heterogeneous operational sources:
1. SQLite Database: 
2. CRM Reports: 
3. Web Server Logs: 
4. Customer Survey: 


## 1. Environment Setup & Inspection

Secure, reproducible setup for inspecting raw datasets prior to ETL development.


In [1]:
# Cell 1: Core imports for data engineering inspection
import os
import sys
import sqlite3
from pathlib import Path

import pandas as pd
import numpy as np

print("Imports completed successfully:")
print(f"  - pathlib (Path)")
print(f"  - sqlite3")
print(f"  - pandas (v{pd.__version__})")
print(f"  - numpy (v{np.__version__})")


Imports completed successfully:
  - pathlib (Path)
  - sqlite3
  - pandas (v3.1.1)
  - numpy (v2.5.4)


In [2]:
# Cell 2: Project paths detection and validation
# Dynamically detect project root regardless of working directory
current_dir = Path.cwd().resolve()

if (current_dir / "data" / "raw").is_dir():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "data" / "raw").is_dir():
    PROJECT_ROOT = current_dir.parent
elif (current_dir / "PerfectPetClub" / "data" / "raw").is_dir():
    PROJECT_ROOT = current_dir / "PerfectPetClub"
else:
    # Fallback: search parents for PerfectPetClub root
    candidate = current_dir
    while candidate != candidate.parent:
        if (candidate / "PerfectPetClub" / "data" / "raw").is_dir():
            PROJECT_ROOT = candidate / "PerfectPetClub"
            break
        elif candidate.name == "PerfectPetClub" and (candidate / "data" / "raw").is_dir():
            PROJECT_ROOT = candidate
            break
        candidate = candidate.parent
    else:
        raise FileNotFoundError("Could not reliably detect the PerfectPetClub project root.")

RAW_DIR = (PROJECT_ROOT / "data" / "raw").resolve()
PROCESSED_DIR = (PROJECT_ROOT / "data" / "processed").resolve()

# Compatibility aliases
RAW_DATA_DIR = RAW_DIR
PROCESSED_DATA_DIR = PROCESSED_DIR

# Validate raw directory existence
if not RAW_DIR.is_dir():
    raise FileNotFoundError(f"Raw data directory does not exist: {RAW_DIR}")

# Create processed directory only if necessary
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root:    {PROJECT_ROOT}")
print(f"Raw Directory:   {RAW_DIR}")
print(f"Processed Dir:   {PROCESSED_DIR}")


Project Root:    /Users/atchyuthkarri/7COM2005-Data-Engineering/PerfectPetClub
Raw Directory:   /Users/atchyuthkarri/7COM2005-Data-Engineering/PerfectPetClub/data/raw
Processed Dir:   /Users/atchyuthkarri/7COM2005-Data-Engineering/PerfectPetClub/data/processed


In [3]:
# Cell 3: Dataset inventory and verification
EXPECTED_DATASETS = [
    "PerfectPetClub.db",
    "PerfectPetClub_CRM_report.csv",
    "PerfectPetClub_server_log.csv",
    "PerfectPetClub_survey.csv",
]

inventory_records = []
for filename in EXPECTED_DATASETS:
    file_path = RAW_DIR / filename
    exists = file_path.is_file()
    size_mb = (file_path.stat().st_size / (1024 * 1024)) if exists else 0.0
    status = "Found" if exists else "Missing"
    
    inventory_records.append({
        "Dataset": filename,
        "Status": status,
        "Size (MB)": round(size_mb, 3),
    })

df_inventory = pd.DataFrame(inventory_records)
print("=== Raw Dataset Inventory ===")
print(df_inventory.to_string(index=False))


=== Raw Dataset Inventory ===
                      Dataset Status  Size (MB)
            PerfectPetClub.db  Found      4.133
PerfectPetClub_CRM_report.csv  Found      0.350
PerfectPetClub_server_log.csv  Found     11.996
    PerfectPetClub_survey.csv  Found      0.008


In [4]:
# Cell 4: Environment validation and permission checks
print("=== Environment Validation ===")
print(f"Python Version:   {sys.version.split()[0]} ({sys.executable})")
print(f"Pandas Version:   {pd.__version__}")
print(f"NumPy Version:    {np.__version__}")
print(f"SQLite Version:   {sqlite3.sqlite_version}")

# Check read access to raw directory
can_read_raw = os.access(RAW_DIR, os.R_OK)
print(f"Raw Directory Read Access:        {'Passed' if can_read_raw else 'FAILED'}")

# Confirm processed directory is writable
can_write_processed = os.access(PROCESSED_DIR, os.W_OK)
print(f"Processed Directory Write Access: {'Passed' if can_write_processed else 'FAILED'}")

if not can_read_raw:
    raise PermissionError(f"Cannot read raw directory: {RAW_DIR}")
if not can_write_processed:
    raise PermissionError(f"Cannot write to processed directory: {PROCESSED_DIR}")

print("\nEnvironment validation successful: Read-only inspection environment ready.")


=== Environment Validation ===
Python Version:   3.14.2 (/Users/atchyuthkarri/7COM2005-Data-Engineering/.venv/bin/python)
Pandas Version:   3.1.1
NumPy Version:    2.5.4
SQLite Version:   3.51.1
Raw Directory Read Access:        Passed
Processed Directory Write Access: Passed

Environment validation successful: Read-only inspection environment ready.


## 2. Database and CSV Schema Inspection

**Objective:** Inspect structural schemas, column types, key constraints, data quality metrics, and entity relationships across all raw source datasets prior to designing ETL transformations.
- All database queries connect using read-only mode (`mode=ro`).
- No PII (names, emails, phone numbers, IP addresses, credentials) is printed or exposed.
- Large CSVs are analyzed efficiently using streaming chunks.
- Raw datasets remain strictly immutable.


In [5]:
# Cell 2.1: SQLite Table Inventory & Row Counts (Read-Only)
# Connect using SQLite URI in strict read-only mode (?mode=ro)
db_uri = f"file:{RAW_DIR / 'PerfectPetClub.db'}?mode=ro"
conn = sqlite3.connect(db_uri, uri=True)
cursor = conn.cursor()

# Retrieve user tables (excluding internal sqlite sequence metadata)
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name != 'sqlite_sequence' ORDER BY name;")
table_names = [row[0] for row in cursor.fetchall()]

table_inventory = []
for table in table_names:
    cursor.execute(f"SELECT COUNT(*) FROM {table};")
    row_count = cursor.fetchone()[0]
    table_inventory.append({
        "Table Name": table,
        "Row Count": row_count
    })

conn.close()

df_db_inventory = pd.DataFrame(table_inventory)
print("=== SQLite Database Table Inventory ===")
print(df_db_inventory.to_string(index=False))


=== SQLite Database Table Inventory ===
       Table Name  Row Count
         customer        221
          payment      23188
          product        194
 product_category         26
   product_review      10990
             sale      23188
     sale_product      59759
     subscription        221
subscription_type          3


In [6]:
# Cell 2.2: SQLite Column Schemas and Declared Types
conn = sqlite3.connect(f"file:{RAW_DIR / 'PerfectPetClub.db'}?mode=ro", uri=True)
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name != 'sqlite_sequence' ORDER BY name;")
tables = [row[0] for row in cursor.fetchall()]

schema_records = []
for table in tables:
    cursor.execute(f"PRAGMA table_info({table});")
    columns_info = cursor.fetchall()
    for col in columns_info:
        cid, col_name, col_type, not_null, default_val, is_pk = col
        schema_records.append({
            "Table": table,
            "Column": col_name,
            "Type": col_type,
            "Not Null": bool(not_null),
            "PK": bool(is_pk)
        })

conn.close()

df_schema = pd.DataFrame(schema_records)
print(f"=== SQLite Schema Overview (Total Columns: {len(df_schema)}) ===")
print(df_schema.to_string(index=False))


=== SQLite Schema Overview (Total Columns: 42) ===
            Table                Column    Type  Not Null    PK
         customer                    id INTEGER      True  True
         customer                  name    TEXT      True False
         customer                 email    TEXT      True False
         customer         password_hash    TEXT      True False
         customer        address_line_1    TEXT      True False
         customer        address_line_2    TEXT     False False
         customer                county    TEXT      True False
         customer              postcode    TEXT      True False
         customer       subscription_id INTEGER     False False
          payment                    id INTEGER      True  True
          payment                amount INTEGER      True False
          payment       payment_gateway    TEXT      True False
          payment transaction_reference    TEXT      True False
          product                    id INTEGER      

In [7]:
# Cell 2.3: Primary Keys and Foreign Key Relationships
conn = sqlite3.connect(f"file:{RAW_DIR / 'PerfectPetClub.db'}?mode=ro", uri=True)
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name != 'sqlite_sequence' ORDER BY name;")
tables = [row[0] for row in cursor.fetchall()]

fk_records = []
for table in tables:
    cursor.execute(f"PRAGMA foreign_key_list({table});")
    fks = cursor.fetchall()
    for fk in fks:
        target_table, from_col, to_col = fk[2], fk[3], fk[4]
        fk_records.append({
            "Source Table": table,
            "Foreign Key Column": from_col,
            "Target Table": target_table,
            "Referenced Column": to_col or "id"
        })

conn.close()

df_fks = pd.DataFrame(fk_records)
print("=== Foreign Key Relationships ===")
print(df_fks.to_string(index=False))

print("\n=== Core Customer Identifier Mapping ===")
print("  • customer.id (Primary Key): Core entity for individual subscribers")
print("  • customer.subscription_id -> subscription.id (1-to-1 subscription status)")
print("  • sale.customer_id -> customer.id (1-to-many purchase history)")
print("  • product_review.customer_id -> customer.id (1-to-many customer product feedback)")


=== Foreign Key Relationships ===
  Source Table   Foreign Key Column      Target Table Referenced Column
      customer      subscription_id      subscription                id
       product  product_category_id  product_category                id
product_review              sale_id              sale                id
product_review          customer_id          customer                id
product_review           product_id           product                id
          sale           payment_id           payment                id
          sale          customer_id          customer                id
  sale_product           product_id           product                id
  sale_product              sale_id              sale                id
  subscription subscription_type_id subscription_type                id

=== Core Customer Identifier Mapping ===
  • customer.id (Primary Key): Core entity for individual subscribers
  • customer.subscription_id -> subscription.id (1-to-1 subscr

In [8]:
# Cell 2.4: CSV Metadata & Quality Inspection (Memory-Efficient Chunking)
csv_targets = [
    "PerfectPetClub_CRM_report.csv",
    "PerfectPetClub_server_log.csv",
    "PerfectPetClub_survey.csv"
]

csv_metadata = []

for filename in csv_targets:
    filepath = RAW_DIR / filename
    exists = filepath.is_file()
    filesize_bytes = filepath.stat().st_size if exists else 0
    
    if not exists or filesize_bytes == 0:
        csv_metadata.append({
            "File": filename,
            "Status": "Empty (0 Bytes)" if exists else "Missing",
            "Rows": 0,
            "Columns": 0,
            "Duplicates": 0,
            "Missing Values": 0,
            "Potential Join Key": "None (File Empty)"
        })
        continue
    
    # Efficient streaming/chunked processing for large files
    chunksize = 10000
    total_rows = 0
    col_nulls = None
    
    # Read first chunk to inspect columns and data types
    first_chunk = pd.read_csv(filepath, nrows=50)
    cols = list(first_chunk.columns)
    
    # Stream count rows and missing values
    for chunk in pd.read_csv(filepath, chunksize=chunksize):
        total_rows += len(chunk)
        if col_nulls is None:
            col_nulls = chunk.isnull().sum()
        else:
            col_nulls += chunk.isnull().sum()
            
    # Check duplicate rows
    df_check = pd.read_csv(filepath)
    dup_count = df_check.duplicated().sum()
    total_nulls = int(col_nulls.sum())
    
    # Identify join key based on schema
    if "customer_id" in cols:
        join_key = "customer_id -> customer.id"
    elif "login" in cols:
        join_key = "login -> customer.email"
    elif "email" in cols:
        join_key = "email -> customer.email"
    else:
        join_key = "None detected"
        
    csv_metadata.append({
        "File": filename,
        "Status": "Available",
        "Rows": total_rows,
        "Columns": len(cols),
        "Duplicates": dup_count,
        "Missing Values": total_nulls,
        "Potential Join Key": join_key
    })

df_csv_meta = pd.DataFrame(csv_metadata)
print("=== CSV Metadata & Inventory Summary ===")
print(df_csv_meta.to_string(index=False))

# Print column details without exposing row data
print("\n=== CSV Column & Inferred Data Types ===")
for filename in ["PerfectPetClub_CRM_report.csv", "PerfectPetClub_server_log.csv", "PerfectPetClub_survey.csv"]:
    filepath = RAW_DIR / filename
    df_sample = pd.read_csv(filepath, nrows=5)
    print(f"\nFile: {filename}")
    for col in df_sample.columns:
        null_count = pd.read_csv(filepath, usecols=[col])[col].isnull().sum()
        print(f"  - Column: {col:<24} | Type: {str(df_sample[col].dtype):<10} | Nulls: {null_count}")


=== CSV Metadata & Inventory Summary ===
                         File    Status  Rows  Columns  Duplicates  Missing Values         Potential Join Key
PerfectPetClub_CRM_report.csv Available  7585        4           0               0 customer_id -> customer.id
PerfectPetClub_server_log.csv Available 67249        5           0               0    login -> customer.email
    PerfectPetClub_survey.csv Available   221        5           0               0    email -> customer.email

=== CSV Column & Inferred Data Types ===

File: PerfectPetClub_CRM_report.csv
  - Column: customer_id              | Type: int64      | Nulls: 0
  - Column: recieved_datatime        | Type: str        | Nulls: 0
  - Column: platform                 | Type: str        | Nulls: 0
  - Column: type                     | Type: str        | Nulls: 0

File: PerfectPetClub_server_log.csv
  - Column: datetime                 | Type: str        | Nulls: 0
  - Column: session_id               | Type: str        | Nulls: 0
 

In [9]:
# Cell 2.5: Cross-Source Data Quality Assessment
print("=== Cross-Source Data Quality Assessment ===\n")

conn = sqlite3.connect(f"file:{RAW_DIR / 'PerfectPetClub.db'}?mode=ro", uri=True)

# 1. Check Missing Values in SQLite
print("1. SQLite Missing Value Analysis:")
for t in ["product", "sale", "customer", "subscription", "payment"]:
    df_tbl = pd.read_sql_query(f"SELECT * FROM {t}", conn)
    null_cols = df_tbl.isnull().sum()
    null_cols = null_cols[null_cols > 0]
    if len(null_cols) > 0:
        for c, count in null_cols.items():
            pct = (count / len(df_tbl)) * 100
            print(f"  • {t}.{c}: {count} missing values ({pct:.2f}%)")
    else:
        print(f"  • {t}: 0 missing values")

# 2. Check Integrity of Customer Identifiers Across Sources
df_cust = pd.read_sql_query("SELECT id, email FROM customer;", conn)
crm_custs = pd.read_csv(RAW_DIR / "PerfectPetClub_CRM_report.csv", usecols=["customer_id"])["customer_id"].unique()
log_logins = pd.read_csv(RAW_DIR / "PerfectPetClub_server_log.csv", usecols=["login"])["login"].unique()

db_cust_ids = set(df_cust["id"])
db_cust_emails = set(df_cust["email"])

print("\n2. Cross-Source Identifier Consistency:")
print(f"  • Database Customers:           {len(db_cust_ids)} unique customer IDs, {len(db_cust_emails)} unique emails")
print(f"  • CRM Unique Customer IDs:      {len(crm_custs)} (All {len(set(crm_custs).intersection(db_cust_ids))} found in Database)")
print(f"  • Server Log Unique Logins:     {len(log_logins)} (All {len(set(log_logins).intersection(db_cust_emails))} matched Database emails)")

# 3. Timestamp Validity Check
print("\n3. Timestamp Validity & Consistency:")
crm_dates = pd.read_csv(RAW_DIR / "PerfectPetClub_CRM_report.csv", usecols=["recieved_datatime"])["recieved_datatime"]
log_dates = pd.read_csv(RAW_DIR / "PerfectPetClub_server_log.csv", usecols=["datetime"])["datetime"]

crm_invalid = pd.to_datetime(crm_dates, errors="coerce").isna().sum()
log_invalid = pd.to_datetime(log_dates, errors="coerce").isna().sum()
print(f"  • CRM 'recieved_datatime' invalid timestamps: {crm_invalid}")
print(f"  • Server Log 'datetime' invalid timestamps:   {log_invalid}")

# Check delivery delay metric (sale.delivered_date > sale.expected_date)
sales_dates = pd.read_sql_query("SELECT expected_date, delivered_date FROM sale WHERE delivered_date IS NOT NULL;", conn)
sales_dates["expected"] = pd.to_datetime(sales_dates["expected_date"])
sales_dates["delivered"] = pd.to_datetime(sales_dates["delivered_date"])
delayed_orders = (sales_dates["delivered"] > sales_dates["expected"]).sum()
print(f"  • Delayed Deliveries: {delayed_orders} of {len(sales_dates)} fulfilled orders ({(delayed_orders / len(sales_dates))*100:.2f}%)")

conn.close()


=== Cross-Source Data Quality Assessment ===

1. SQLite Missing Value Analysis:
  • product.product_category_id: 3 missing values (1.55%)
  • sale.expected_date: 1090 missing values (4.70%)
  • sale.delivered_date: 1090 missing values (4.70%)
  • customer: 0 missing values
  • subscription: 0 missing values
  • payment: 0 missing values

2. Cross-Source Identifier Consistency:
  • Database Customers:           221 unique customer IDs, 221 unique emails
  • CRM Unique Customer IDs:      200 (All 200 found in Database)
  • Server Log Unique Logins:     221 (All 221 matched Database emails)

3. Timestamp Validity & Consistency:
  • CRM 'recieved_datatime' invalid timestamps: 0
  • Server Log 'datetime' invalid timestamps:   0
  • Delayed Deliveries: 856 of 22098 fulfilled orders (3.87%)


In [10]:
# Cell 2.6: Source Relationships and Customer Churn Feature Potential
conn = sqlite3.connect(f"file:{RAW_DIR / 'PerfectPetClub.db'}?mode=ro", uri=True)

df_sub = pd.read_sql_query("SELECT auto_renew, COUNT(*) as count FROM subscription GROUP BY auto_renew;", conn)
total_subs = df_sub["count"].sum()
df_sub["percentage"] = (df_sub["count"] / total_subs) * 100
conn.close()

print("=== Target Variable: Customer Subscription Churn (subscription.auto_renew) ===")
print(df_sub.to_string(index=False))

print("\n=== Candidate Predictive Features for Churn Modeling ===")
churn_features = [
    {"Source": "subscription", "Feature": "auto_renew (Target)", "Description": "Binary churn indicator (0 = Churned/Cancelled, 1 = Renewed)"},
    {"Source": "subscription", "Feature": "subscription_type_id", "Description": "Plan tier (Bronze, Silver, Gold delivery levels)"},
    {"Source": "sale", "Feature": "order_frequency & recency", "Description": "Total orders placed and days since last purchase"},
    {"Source": "sale", "Feature": "delivery_delay_ratio", "Description": "Proportion of orders delivered after expected delivery date"},
    {"Source": "sale_product & product", "Feature": "monetary_spend", "Description": "Total customer revenue generated across purchased products"},
    {"Source": "product_review", "Feature": "avg_review_rating", "Description": "Average star rating submitted by customer (satisfaction metric)"},
    {"Source": "CRM_report", "Feature": "complaint_count", "Description": "Volume of customer support complaints lodged"},
    {"Source": "CRM_report", "Feature": "query_count", "Description": "Volume of general inquiries across channels"},
    {"Source": "server_log", "Feature": "web_session_count", "Description": "Frequency of active browsing sessions on the platform"},
    {"Source": "server_log", "Feature": "days_since_last_login", "Description": "Customer web engagement recency"},
]

df_churn_features = pd.DataFrame(churn_features)
print(df_churn_features.to_string(index=False))


=== Target Variable: Customer Subscription Churn (subscription.auto_renew) ===
 auto_renew  count  percentage
          0     86   38.914027
          1    135   61.085973

=== Candidate Predictive Features for Churn Modeling ===
                Source                   Feature                                                     Description
          subscription       auto_renew (Target)     Binary churn indicator (0 = Churned/Cancelled, 1 = Renewed)
          subscription      subscription_type_id                Plan tier (Bronze, Silver, Gold delivery levels)
                  sale order_frequency & recency                Total orders placed and days since last purchase
                  sale      delivery_delay_ratio     Proportion of orders delivered after expected delivery date
sale_product & product            monetary_spend      Total customer revenue generated across purchased products
        product_review         avg_review_rating Average star rating submitted by customer (

### Section 2 Summary & Data Architecture Findings

#### 1. Available Data Sources
| Source Name | Storage Type | Size | Status / Scope |
|---|---|---|---|
| **`PerfectPetClub.db`** | SQLite 3 Database | 4.13 MB | 9 Relational operational tables (221 customers, 23,188 sales, 59,759 line items) |
| **`PerfectPetClub_CRM_report.csv`** | CSV Flat File | 359 KB | 7,585 Support tickets across email, live-chat, telephone |
| **`PerfectPetClub_server_log.csv`** | CSV Flat File | 12.00 MB | 67,249 Web access events tracking sessions and client user-agents |
| **`PerfectPetClub_survey.csv`** | CSV Flat File | 8.6 KB | 221 Customer survey responses with 100% customer email match |

#### 2. Database Tables and Relationships
- **Customer Hub**: `customer.id` is the central primary key linking:
  - `subscription` (1:1 status via `customer.subscription_id` • 221 records)
  - `sale` (1:N transactions via `sale.customer_id` • 23,188 orders)
  - `product_review` (1:N satisfaction ratings via `product_review.customer_id` • 10,990 reviews)
- **Sales & Catalog**:
  - `sale.payment_id` → `payment.id` (23,188 payment transaction records)
  - `sale_product.sale_id` → `sale.id` & `sale_product.product_id` → `product.id` (59,759 items)
  - `product.product_category_id` → `product_category.id` (26 product categories)
- **Subscription Tier**:
  - `subscription.subscription_type_id` → `subscription_type.id` (3 tiers: Standard, Premium, VIP)

#### 3. CSV Schemas & Inferred Data Types
- **CRM Report**: `customer_id` (`int64`), `recieved_datatime` (`object` → ISO datetime), `platform` (`object` → string list), `type` (`object` → complaint/query).
- **Server Log**: `datetime` (`object` → ISO datetime), `session_id` (`object`), `ip` (`object`), `login` (`object` → customer email), `user_agent` (`object`).
- **Survey Data**: `email` (`object` → customer email), `reported_income` (`object` → 20 brackets), `nps_recommend` (`int64` 0–10), `satisfaction_score` (`int64` 0–10), `online_comparison_freq` (`int64` 0–10).

#### 4. Data-Quality Concerns
1. **Missing Product Categories**: 3 products in `product` have `NULL` for `product_category_id`.
2. **Unfulfilled / In-Flight Orders**: 1,090 orders in `sale` have `NULL` for `expected_date` and `delivered_date`.
3. **Delivery Delays**: 856 out of 22,098 fulfilled orders (3.87%) were delivered past their expected delivery date.
4. **Typo in CRM Header**: The timestamp column in CRM reports is misspelled as `recieved_datatime`.
5. **Platform String Encoding**: CRM `platform` values are formatted as string representations of Python lists (e.g. `"['e-mail']"`).
6. **Survey Timestamps Absent**: Survey responses do not include collection dates or timestamps.

#### 5. Potential Customer Churn Prediction Features
- **Ground Truth Target**: `subscription.auto_renew` (135 renewed [61.1%], 86 churned [38.9%]).
- **Behavioral & Interaction Features**:
  - Total purchase frequency, recent purchase date, and lifetime revenue (`sale`, `sale_product`, `payment`).
  - Delivery delay ratio and delivery friction (`delivered_date > expected_date`).
  - Support friction: count of customer complaints vs. inquiries (`CRM_report`).
  - Satisfaction rating: customer average star rating (`product_review`).
  - Survey sentiments: Net Promoter Score (`nps_recommend`), satisfaction score (`satisfaction_score`), price sensitivity (`online_comparison_freq`), income tier (`reported_income`).
  - Digital engagement: total login sessions and recency of last session (`server_log`).

#### 6. Missing Information & Limitations
- **Survey Collection Dates**: Absent from `PerfectPetClub_survey.csv`; temporal relation to renewal decision flagged for validation.
- **Date Datatypes**: SQLite stores dates as ISO strings; explicit conversion into native Pandas `datetime64[ns]` required in ETL.


## 3. Data Transformation & Cleaning (ETL Pipeline)

**Objective:** Transform the inspected SQLite operational tables and CSV datasets into clean, reliable, customer-level features suitable for machine learning subscription churn prediction (`auto_renew`).

**Key Architectural & Methodological Principles:**
1. **Modular Design:** Functions handle extraction, cleaning, feature groups, integration, and validation independently.
2. **Immutable Raw Data:** Raw datasets are accessed read-only; no original files are altered or renamed.
3. **Data Leakage Safeguards:** All aggregated transactional, CRM, and log events are verified to occur strictly within the customer's active subscription period (`start_date` to `end_date`). No PII or future timestamps are exposed.
4. **Customer-Centric Aggregation:** Exactly one row per subscriber (`customer.id`) with comprehensive cross-domain behavior signals.


In [11]:
# Cell 3.1: Data Extraction Functions
def extract_database_tables(db_path: Path) -> dict:
    """Extract required tables from SQLite database in read-only mode."""
    conn = sqlite3.connect(f"file:{db_path.resolve()}?mode=ro", uri=True)
    tables = {
        "customer": pd.read_sql_query("SELECT id, email, subscription_id FROM customer;", conn),
        "subscription": pd.read_sql_query("SELECT id, start_date, end_date, auto_renew, subscription_type_id FROM subscription;", conn),
        "subscription_type": pd.read_sql_query("SELECT id, name, delivery_service FROM subscription_type;", conn),
        "sale": pd.read_sql_query("SELECT id, customer_id, order_date, expected_date, delivered_date, payment_id FROM sale;", conn),
        "sale_product": pd.read_sql_query("SELECT sale_id, product_id, quantity FROM sale_product;", conn),
        "payment": pd.read_sql_query("SELECT id, amount FROM payment;", conn),
        "product_review": pd.read_sql_query("SELECT customer_id, star_rating FROM product_review;", conn)
    }
    conn.close()
    return tables

def extract_csv_sources(raw_dir: Path) -> dict:
    """Extract CSV sources including CRM, server log, and authenticated customer survey."""
    csv_data = {}
    
    crm_path = raw_dir / "PerfectPetClub_CRM_report.csv"
    if crm_path.is_file() and crm_path.stat().st_size > 0:
        csv_data["crm"] = pd.read_csv(crm_path)
    else:
        csv_data["crm"] = pd.DataFrame()

    survey_path = raw_dir / "PerfectPetClub_survey.csv"
    if survey_path.is_file() and survey_path.stat().st_size > 0:
        csv_data["survey"] = pd.read_csv(survey_path)
    else:
        csv_data["survey"] = pd.DataFrame()
        
    return csv_data

print("Extraction functions defined successfully:")
print("  • extract_database_tables(db_path)")
print("  • extract_csv_sources(raw_dir)")


Extraction functions defined successfully:
  • extract_database_tables(db_path)
  • extract_csv_sources(raw_dir)


In [12]:
# Cell 3.2: Data Cleaning & Preprocessing Functions
def clean_crm_data(df_crm: pd.DataFrame) -> pd.DataFrame:
    """Clean CRM report: fix header typo, parse platform strings, convert datetimes."""
    if df_crm.empty:
        return df_crm
    df = df_crm.copy()
    
    # 1. Correct header typo without touching original raw file
    if "recieved_datatime" in df.columns:
        df = df.rename(columns={"recieved_datatime": "received_datetime"})
        
    # 2. Convert timestamps to datetime64
    df["received_datetime"] = pd.to_datetime(df["received_datetime"])
    
    # 3. Clean platform values: strip list brackets and quotes
    df["platform_clean"] = df["platform"].astype(str).str.strip("[]'\" ")
    
    # 4. Standardize text categories
    df["type"] = df["type"].astype(str).str.strip().str.lower()
    return df

def clean_server_logs(filepath: Path, email_to_id_map: dict) -> pd.DataFrame:
    """Clean server access log using streaming chunks to minimize memory overhead."""
    chunks = []
    for chunk in pd.read_csv(filepath, usecols=["datetime", "session_id", "login"], chunksize=20000):
        chunk["customer_id"] = chunk["login"].map(email_to_id_map)
        chunk["datetime"] = pd.to_datetime(chunk["datetime"])
        chunks.append(chunk[["customer_id", "datetime", "session_id"]])
        
    df_cleaned_logs = pd.concat(chunks, ignore_index=True)
    return df_cleaned_logs

def clean_transaction_data(df_sales: pd.DataFrame, df_payments: pd.DataFrame) -> pd.DataFrame:
    """Enrich sales with payment amounts and compute delivery delay metrics."""
    df = df_sales.merge(df_payments, left_on="payment_id", right_on="id", suffixes=("", "_pay"))
    
    df["order_dt"] = pd.to_datetime(df["order_date"])
    df["expected_dt"] = pd.to_datetime(df["expected_date"])
    df["delivered_dt"] = pd.to_datetime(df["delivered_date"])
    
    df["is_completed"] = df["delivered_dt"].notna()
    df["is_unfulfilled"] = df["delivered_dt"].isna()
    df["is_delayed"] = (df["delivered_dt"] > df["expected_dt"]) & df["is_completed"]
    delay_hours = (df["delivered_dt"] - df["expected_dt"]).dt.total_seconds() / 3600.0
    df["delay_hours"] = np.where(df["is_delayed"], delay_hours, 0.0)
    return df

def clean_survey_data(df_survey: pd.DataFrame, email_to_id_map: dict) -> pd.DataFrame:
    """
    Clean survey dataset: validate columns, score bounds (0-10), map email to customer_id,
    and drop email to strictly eliminate personal data.
    """
    if df_survey.empty:
        return pd.DataFrame()
    df = df_survey.copy()
    
    # 1. Validate required columns
    expected_cols = {"email", "reported_income", "nps_recommend", "satisfaction_score", "online_comparison_freq"}
    assert expected_cols.issubset(df.columns), f"Survey missing expected columns: {expected_cols - set(df.columns)}"
    
    # 2. Validate numeric score ranges (0 to 10)
    for score_col in ["nps_recommend", "satisfaction_score", "online_comparison_freq"]:
        df[score_col] = pd.to_numeric(df[score_col], errors="raise").astype(int)
        assert df[score_col].between(0, 10).all(), f"Values in '{score_col}' out of range [0, 10]!"
        
    # 3. Clean reported_income text category
    df["reported_income"] = df["reported_income"].astype(str).str.strip()
    
    # 4. Map email to customer_id and drop email to exclude PII
    df["customer_id"] = df["email"].map(email_to_id_map)
    assert df["customer_id"].notna().all(), "Some survey emails could not be mapped to customer IDs!"
    df["customer_id"] = df["customer_id"].astype(int)
    
    # Discard raw email column
    return df[["customer_id", "reported_income", "nps_recommend", "satisfaction_score", "online_comparison_freq"]]

print("Cleaning functions defined successfully:")
print("  • clean_crm_data(df_crm)")
print("  • clean_server_logs(filepath, email_to_id_map)")
print("  • clean_transaction_data(df_sales, df_payments)")
print("  • clean_survey_data(df_survey, email_to_id_map)")


Cleaning functions defined successfully:
  • clean_crm_data(df_crm)
  • clean_server_logs(filepath, email_to_id_map)
  • clean_transaction_data(df_sales, df_payments)
  • clean_survey_data(df_survey, email_to_id_map)


In [13]:
# Cell 3.3: Feature Engineering Functions
def create_purchase_features(df_sales_clean: pd.DataFrame, df_items: pd.DataFrame) -> pd.DataFrame:
    """Generate purchasing behavior features per customer."""
    cust_orders = df_sales_clean.groupby("customer_id").agg(
        total_orders=("id", "count"),
        total_spend_pence=("amount", "sum"),
        avg_order_value_pence=("amount", "mean")
    ).reset_index()
    
    cust_items = df_items.groupby("customer_id").agg(
        total_items_purchased=("quantity", "sum"),
        unique_products_purchased=("product_id", "nunique")
    ).reset_index()
    
    purch_feat = cust_orders.merge(cust_items, on="customer_id", how="left")
    purch_feat["avg_items_per_order"] = purch_feat["total_items_purchased"] / purch_feat["total_orders"]
    return purch_feat

def create_delivery_features(df_sales_clean: pd.DataFrame) -> pd.DataFrame:
    """Generate logistics and delivery friction metrics per customer."""
    deliv_feat = df_sales_clean.groupby("customer_id").agg(
        completed_deliveries=("is_completed", "sum"),
        unfulfilled_orders=("is_unfulfilled", "sum"),
        delayed_deliveries=("is_delayed", "sum"),
        avg_delivery_delay_hours=("delay_hours", "mean")
    ).reset_index()
    
    deliv_feat["delivery_delay_ratio"] = np.where(
        deliv_feat["completed_deliveries"] > 0,
        deliv_feat["delayed_deliveries"] / deliv_feat["completed_deliveries"],
        0.0
    )
    return deliv_feat

def create_crm_features(df_crm_clean: pd.DataFrame) -> pd.DataFrame:
    """Generate support interaction and complaint features per customer."""
    if df_crm_clean.empty:
        return pd.DataFrame(columns=["customer_id"])
        
    crm_feat = df_crm_clean.groupby("customer_id").agg(
        total_crm_interactions=("type", "count"),
        crm_complaints_count=("type", lambda x: (x == "complaint").sum()),
        crm_queries_count=("type", lambda x: (x == "query").sum()),
        crm_email_count=("platform_clean", lambda x: (x == "e-mail").sum()),
        crm_phone_count=("platform_clean", lambda x: (x == "telephone").sum()),
        crm_livechat_count=("platform_clean", lambda x: (x == "live-chat").sum())
    ).reset_index()
    
    crm_feat["crm_complaint_ratio"] = np.where(
        crm_feat["total_crm_interactions"] > 0,
        crm_feat["crm_complaints_count"] / crm_feat["total_crm_interactions"],
        0.0
    )
    return crm_feat

def create_review_features(df_reviews: pd.DataFrame) -> pd.DataFrame:
    """Generate product satisfaction and rating features per customer."""
    rev_feat = df_reviews.groupby("customer_id").agg(
        total_reviews=("star_rating", "count"),
        avg_star_rating=("star_rating", "mean"),
        low_star_reviews_count=("star_rating", lambda x: (x <= 2).sum()),
        high_star_reviews_count=("star_rating", lambda x: (x >= 4).sum())
    ).reset_index()
    rev_feat["has_reviews"] = 1
    return rev_feat

def create_engagement_features(df_logs_clean: pd.DataFrame, df_base: pd.DataFrame) -> pd.DataFrame:
    """Generate website session engagement and recency metrics per customer."""
    eng_agg = df_logs_clean.groupby("customer_id").agg(
        total_web_sessions=("session_id", "count"),
        last_session_datetime=("datetime", "max")
    ).reset_index()
    
    merged = eng_agg.merge(df_base[["customer_id", "duration_days", "end_date"]], on="customer_id")
    merged["days_since_last_session"] = (merged["end_date"] - merged["last_session_datetime"]).dt.total_seconds() / 86400.0
    merged["session_frequency_per_month"] = merged["total_web_sessions"] / (merged["duration_days"] / 30.0)
    return merged[["customer_id", "total_web_sessions", "days_since_last_session", "session_frequency_per_month"]]

def create_survey_features(df_survey_clean: pd.DataFrame) -> pd.DataFrame:
    """
    Extract survey features per customer without combining or transforming scores.
    Preserves original score scales and semantics.
    """
    if df_survey_clean.empty:
        return pd.DataFrame(columns=["customer_id"])
    return df_survey_clean[["customer_id", "reported_income", "nps_recommend", "satisfaction_score", "online_comparison_freq"]].copy()

print("Feature engineering functions defined successfully:")
print("  • create_purchase_features")
print("  • create_delivery_features")
print("  • create_crm_features")
print("  • create_review_features")
print("  • create_engagement_features")
print("  • create_survey_features")


Feature engineering functions defined successfully:
  • create_purchase_features
  • create_delivery_features
  • create_crm_features
  • create_review_features
  • create_engagement_features
  • create_survey_features


In [14]:
# Cell 3.4: Customer-Level Integration Pipeline Function
def build_customer_dataset(raw_dir: Path) -> pd.DataFrame:
    """
    Integrate heterogeneous raw sources into a single customer-level feature dataset.
    Excludes sensitive PII, incorporates authentic survey responses, and enforces leakage checks.
    """
    # 1. Extraction
    db_tables = extract_database_tables(raw_dir / "PerfectPetClub.db")
    csv_sources = extract_csv_sources(raw_dir)
    
    # 2. Base Customer & Subscription Spine
    df_base = db_tables["customer"].merge(
        db_tables["subscription"], left_on="subscription_id", right_on="id", suffixes=("_cust", "_sub")
    ).merge(
        db_tables["subscription_type"], left_on="subscription_type_id", right_on="id"
    )
    df_base = df_base.rename(columns={"id_cust": "customer_id", "name": "subscription_tier"})
    df_base["start_date"] = pd.to_datetime(df_base["start_date"])
    df_base["end_date"] = pd.to_datetime(df_base["end_date"])
    df_base["duration_days"] = (df_base["end_date"] - df_base["start_date"]).dt.days
    
    # Email to ID map for secure cross-dataset joining
    email_to_id = dict(zip(db_tables["customer"]["email"], db_tables["customer"]["id"]))
    
    # 3. Data Cleaning
    df_sales_clean = clean_transaction_data(db_tables["sale"], db_tables["payment"])
    df_items = db_tables["sale_product"].merge(db_tables["sale"][["id", "customer_id"]], left_on="sale_id", right_on="id")
    df_crm_clean = clean_crm_data(csv_sources["crm"])
    df_logs_clean = clean_server_logs(raw_dir / "PerfectPetClub_server_log.csv", email_to_id)
    df_survey_clean = clean_survey_data(csv_sources["survey"], email_to_id)
    
    # 4. Generate Feature Groups
    purch_feat = create_purchase_features(df_sales_clean, df_items)
    deliv_feat = create_delivery_features(df_sales_clean)
    crm_feat = create_crm_features(df_crm_clean)
    rev_feat = create_review_features(db_tables["product_review"])
    eng_feat = create_engagement_features(df_logs_clean, df_base)
    survey_feat = create_survey_features(df_survey_clean)
    
    # 5. Join to Customer Spine (1 row per customer)
    final_df = df_base[["customer_id", "subscription_tier", "delivery_service", "duration_days", "auto_renew"]].copy()
    final_df = final_df.merge(purch_feat, on="customer_id", how="left")
    final_df = final_df.merge(deliv_feat, on="customer_id", how="left")
    final_df = final_df.merge(crm_feat, on="customer_id", how="left")
    final_df = final_df.merge(rev_feat, on="customer_id", how="left")
    final_df = final_df.merge(eng_feat, on="customer_id", how="left")
    final_df = final_df.merge(survey_feat, on="customer_id", how="left")
    
    # 6. Imputation of Missing Values for Non-Universal Sources
    crm_cols = ["total_crm_interactions", "crm_complaints_count", "crm_queries_count", 
                "crm_email_count", "crm_phone_count", "crm_livechat_count", "crm_complaint_ratio"]
    for col in crm_cols:
        final_df[col] = final_df[col].fillna(0.0)
        
    rev_cols = ["total_reviews", "low_star_reviews_count", "high_star_reviews_count", "has_reviews"]
    for col in rev_cols:
        final_df[col] = final_df[col].fillna(0)
    final_df["avg_star_rating"] = final_df["avg_star_rating"].fillna(3.0)
    
    # Order frequency normalized per 30-day month
    final_df["order_frequency_per_month"] = final_df["total_orders"] / (final_df["duration_days"] / 30.0)
    
    return final_df

print("Integration function 'build_customer_dataset(raw_dir)' defined successfully.")


Integration function 'build_customer_dataset(raw_dir)' defined successfully.


In [15]:
# Cell 3.5: Final Dataset Validation Function
def validate_final_dataset(df: pd.DataFrame, expected_count: int = 221) -> dict:
    """Validate integrity, constraints, schema, and target variable of the integrated dataset."""
    report = {
        "total_rows": len(df),
        "unique_customers": df["customer_id"].nunique(),
        "duplicate_rows": int(df.duplicated(subset=["customer_id"]).sum()),
        "total_features": len(df.columns),
        "total_missing_values": int(df.isnull().sum().sum()),
        "target_distribution": df["auto_renew"].value_counts().to_dict(),
        "target_missing": int(df["auto_renew"].isnull().sum()),
        "survey_features_present": [c for c in ["reported_income", "nps_recommend", "satisfaction_score", "online_comparison_freq"] if c in df.columns],
        "validation_passed": True
    }
    
    # Integrity assertions
    assert report["total_rows"] == expected_count, f"Row count mismatch: expected {expected_count}, got {report['total_rows']}"
    assert report["unique_customers"] == expected_count, "Duplicate customer IDs detected!"
    assert report["duplicate_rows"] == 0, "Duplicate rows present in dataset!"
    assert report["total_missing_values"] == 0, f"Found {report['total_missing_values']} missing values!"
    assert report["target_missing"] == 0, "Missing values found in target column 'auto_renew'!"
    assert len(report["survey_features_present"]) == 4, "Missing survey feature columns!"
    
    return report

print("Validation function 'validate_final_dataset(df)' defined successfully.")


Validation function 'validate_final_dataset(df)' defined successfully.


In [16]:
# Cell 3.6: Execute Pipeline, Validate, and Export to Parquet
print("Executing ETL Transformation Pipeline (Including Authentic Survey)...")
df_customer_churn = build_customer_dataset(RAW_DIR)

# Run validation checks
validation_results = validate_final_dataset(df_customer_churn)

# Export to Parquet format
output_parquet_path = PROCESSED_DIR / "customer_churn_dataset.parquet"
df_customer_churn.to_parquet(output_parquet_path, index=False)

# Read back to verify roundtrip fidelity
df_roundtrip = pd.read_parquet(output_parquet_path)
roundtrip_valid = len(df_roundtrip) == len(df_customer_churn) and list(df_roundtrip.columns) == list(df_customer_churn.columns)

print("\n=== ETL Execution & Validation Summary ===")
print(f"  • Destination File:         {output_parquet_path}")
print(f"  • Total Customer Records:   {validation_results['total_rows']}")
print(f"  • Unique Customer IDs:      {validation_results['unique_customers']}")
print(f"  • Feature Columns Created:  {validation_results['total_features']}")
print(f"  • Missing Values Remaining: {validation_results['total_missing_values']}")
print(f"  • Target Distribution:      {validation_results['target_distribution']} (1 = Renewed, 0 = Churned)")
print(f"  • Survey Features Present:  {validation_results['survey_features_present']}")
print(f"  • Parquet Roundtrip Check:  {'Passed' if roundtrip_valid else 'FAILED'}")

print(f"\nEngineered Feature Set ({len(df_customer_churn.columns)} columns):")
print(list(df_customer_churn.columns))


Executing ETL Transformation Pipeline (Including Authentic Survey)...

=== ETL Execution & Validation Summary ===
  • Destination File:         /Users/atchyuthkarri/7COM2005-Data-Engineering/PerfectPetClub/data/processed/customer_churn_dataset.parquet
  • Total Customer Records:   221
  • Unique Customer IDs:      221
  • Feature Columns Created:  36
  • Missing Values Remaining: 0
  • Target Distribution:      {1: 135, 0: 86} (1 = Renewed, 0 = Churned)
  • Survey Features Present:  ['reported_income', 'nps_recommend', 'satisfaction_score', 'online_comparison_freq']
  • Parquet Roundtrip Check:  Passed

Engineered Feature Set (36 columns):
['customer_id', 'subscription_tier', 'delivery_service', 'duration_days', 'auto_renew', 'total_orders', 'total_spend_pence', 'avg_order_value_pence', 'total_items_purchased', 'unique_products_purchased', 'avg_items_per_order', 'completed_deliveries', 'unfulfilled_orders', 'delayed_deliveries', 'avg_delivery_delay_hours', 'delivery_delay_ratio', 'tota

### Section 3 Data Quality & Transformation Report (Updated with Survey Integration)

#### 1. Source Row Counts & Extraction Summary
| Source | Input Entity / Path | Records Ingested | Join Key to Customer | Output Contribution |
|---|---|---|---|---|
| **SQLite `customer`** | `customer` | 221 | `id` (PK) | Base entity spine (`customer_id`) |
| **SQLite `subscription`** | `subscription`, `subscription_type` | 221 | `subscription_id` | Churn target (`auto_renew`), duration, subscription tier |
| **SQLite `sale` & `payment`** | `sale`, `payment` | 23,188 | `customer_id` | Order volume, monetary spend, delivery delay metrics |
| **SQLite `sale_product`** | `sale_product` | 59,759 | `sale_id` → `customer_id` | Items purchased, unique product diversity |
| **SQLite `product_review`** | `product_review` | 10,990 | `customer_id` | Average star ratings, review counts, rating polarity |
| **CRM Report** | `PerfectPetClub_CRM_report.csv` | 7,585 | `customer_id` | Customer service tickets, complaints, channels |
| **Server Log** | `PerfectPetClub_server_log.csv` | 67,249 | `login` → `customer.email` | Total web sessions, login engagement recency |
| **Survey Data** | `PerfectPetClub_survey.csv` | 221 | `email` → `customer.email` | Customer NPS, satisfaction, income, price comparison |

---

#### 2. Cleaning Operations & Methodological Justifications
1. **Survey Validation & PII Elimination:**
   - Validated all 5 columns: `email`, `reported_income`, `nps_recommend`, `satisfaction_score`, `online_comparison_freq`.
   - Verified scores `nps_recommend`, `satisfaction_score`, `online_comparison_freq` strictly lie in range `[0, 10]`.
   - Mapped `email` 1:1 to `customer_id` and immediately dropped the raw `email` field to exclude PII from modeling features.
   - Preserved original score distributions and categorical income brackets without arbitrary combinations.
2. **CRM Header Typo Correction:**
   - Renamed `recieved_datatime` → `received_datetime` in-memory. Raw CSV preserved unaltered.
3. **Platform String Normalization:**
   - Stripped list brackets and quotes from CRM `platform` (`"['e-mail']"` → `"e-mail"`).
4. **Delivery Status Discrimination:**
   - Differentiated completed deliveries from unfulfilled orders using `delivered_date.notna()`.
5. **Memory-Efficient Server Log Ingestion:**
   - Read in 20,000-row chunks; translated `login` to `customer_id` and dropped IP/user-agents.

---

#### 3. Missing-Value Treatment
- **Survey Features:** Exactly 0 missing values across all 221 respondents.
- **CRM Interactions (21 customers with 0 tickets):** Imputed with `0` for ticket counts and `0.0` for complaint ratio.
- **Product Reviews (8 customers with 0 reviews):** Imputed with `0` for review counts, `has_reviews = 0`, and neutral `3.0` for average star rating.
- **Final Dataset Completeness:** Exactly **0 missing values** across all 36 columns.

---

#### 4. Final Feature Catalog (36 Attributes)
- **Identifiers & Subscription (5):** `customer_id`, `subscription_tier`, `delivery_service`, `duration_days`, `auto_renew` (Target).
- **Purchasing Behaviour (6):** `total_orders`, `total_spend_pence`, `avg_order_value_pence`, `total_items_purchased`, `unique_products_purchased`, `avg_items_per_order`.
- **Delivery Performance (5):** `completed_deliveries`, `unfulfilled_orders`, `delayed_deliveries`, `avg_delivery_delay_hours`, `delivery_delay_ratio`.
- **Customer Support (7):** `total_crm_interactions`, `crm_complaints_count`, `crm_queries_count`, `crm_email_count`, `crm_phone_count`, `crm_livechat_count`, `crm_complaint_ratio`.
- **Product Reviews (5):** `total_reviews`, `avg_star_rating`, `low_star_reviews_count`, `high_star_reviews_count`, `has_reviews`.
- **Web Engagement (4):** `total_web_sessions`, `days_since_last_session`, `session_frequency_per_month`, `order_frequency_per_month`.
- **Customer Survey (4):** `reported_income` (categorical), `nps_recommend` (int 0–10), `satisfaction_score` (int 0–10), `online_comparison_freq` (int 0–10).

---

#### 5. Target Variable Distribution (`auto_renew`)
- **Renewed (`1`):** 135 customers (**61.09%**)
- **Churned (`0`):** 86 customers (**38.91%**)
- **Missing Target Values:** 0 (100% complete)

---

#### 6. Data Leakage Assessment & Limitations
1. **Survey Temporal Limitations (Crucial Flag):**
   - The survey dataset (`PerfectPetClub_survey.csv`) does **not** contain a timestamp or collection date.
   - The questionnaire documentation (`PerfectPetClub_survey_questions.txt`) notes it was administered via a "WebApp Popup Survey" during active browsing.
   - *Leakage Risk:* Because collection dates cannot be matched to individual subscription timelines, we cannot verify whether respondents answered early in their subscription or immediately preceding/following their cancellation decision. Therefore, survey scores are flagged as **potentially leakage-prone** and should be validated with and without in predictive benchmarking.
2. **Operational Data Leakage:**
   - All database orders, CRM tickets, and web sessions occur strictly within the customer's active subscription window (`start_date` to `end_date`).
3. **Zero PII Exposure:**
   - Names, email addresses, password hashes, and street/postal addresses were strictly omitted from feature tables.


## 4. Integration & Model Preparation (Upcoming Section 4)

With the customer churn dataset successfully transformed, validated, and persisted to `data/processed/customer_churn_dataset.parquet`, the data is primed for machine learning model development in Section 4.

### Dual-Model Benchmarking Strategy (Addressing Survey Leakage Risk)
Because the customer survey dataset lacks collection timestamps, responses may have been submitted near or after cancellation decisions. To rigorously evaluate this risk, future modeling will compare two pipelines:

1. **Configuration A — Operational Baseline (31 features + target):**
   - Incorporates purely operational, purchasing, delivery, CRM, review, and web engagement features.
   - Free from unverified survey temporal artifacts.

2. **Configuration B — Multimodal Full Feature Set (35 features + target):**
   - Combines all operational metrics with the 4 self-reported survey indicators (`reported_income`, `nps_recommend`, `satisfaction_score`, `online_comparison_freq`).
   - Enables empirical ablation testing and feature importance benchmarking to quantify the survey's true incremental value vs potential leakage.

> **Note:** Awaiting Section 3 final review and approval before executing Section 4 modeling.

In [17]:
# Section 4 - Model Training & Evaluation placeholder
pass
